# RMA CRAS - Pipeline de Tratamento (Revisado)

Pipeline de ETL dos Relatórios Mensais de Atendimentos (RMA) dos CRAS de Osasco.

Lê o CSV bruto, expande colunas JSON, normaliza nomes, e gera bases analíticas

Para indicadores, raça, identidade de gênero e bairros.

In [1]:
import pandas as pd
import numpy as np
import csv
import json
import glob as glob_module
from pandas import json_normalize
from unidecode import unidecode
import os

StatementMeta(, d2a22d00-e133-4105-a6c5-c1b29694f76b, 3, Finished, Available, Finished, False)

## Configuração

Constantes e mapeamentos utilizados em todo o pipeline.

In [2]:
COLS_ID = [
    "ano",
    "mes",
    "sigla_da_unidade",
]

COLS_BAIRROS = [
    "adalgisa", "alianca", "ayrosa", "bandeiras", "baronesa",
    "bela_vista", "bonanca", "bonfim", "bussocaba", "campesina",
    "castelo_branco", "centro", "cidade_das_flores", "cidade_de_deus",
    "cipava", "city_bussocaba", "conceicao", "helena_maria",
    "industrial_altino", "industrial_anhanguera", "industrial_autonomistas",
    "industrial_centro", "industrial_mazzei", "industrial_remedios",
    "jaguaribe", "jardim_das_flores", "jardim_d_abril", "jardim_elvira",
    "jardim_roberto", "km_18", "metalurgicos", "munhoz_junior",
    "mutinga", "novo_osasco", "outros_municipios", "padroeira",
    "paiva_ramos", "parque_continental", "pestana", "piratininga",
    "platina", "portal_d_oeste", "presidente_altino", "quitauna",
    "raposo_tavares", "remedios", "rochdale", "santa_fe", "santa_maria",
    "santo_antonio", "sao_pedro", "setor_militar", "tres_montanhas",
    "umuarama", "veloso", "vila_menck", "vila_militar", "vila_osasco",
    "vila_yara", "vila_yolanda",
]

COLS_RACA = [
    "e_1_preta_masc", "e_1_preta_fem",
    "e_1_parda_mas", "e_1_parda_fem",
    "e_1_amarela_masc", "e_1_amarela_fem",
    "e_1_indigena_masc", "e_1_indigena_fem",
    "e_1_branca_mas", "e_1_branca_fem",
]

COLS_ID_GENERO = [
    "homem_cis", "homem_trans", "intersexual",
    "mulher_cis", "mulher_trans", "nao_binario",
    "nao_declarado", "travesti",
]

MAP_SIGLA_UNIDADE = {
    "CRAS - Santo Antonio": "CRAS - Santo Antonio",
    "CRAS - 1º DE MAIO": "CRAS - 1º de Maio",
    "CRAS - Veloso": "CRAS - Veloso",
    "CRAS - ROCHDALE": "CRAS - Rochdale",
    "CRAS - Padroeira": "CRAS - Padroeira",
    "CRAS - Bonança": "CRAS - Bonança",
    "CRAS - KM 18": "CRAS - Km 18",
    "CRAS - Piratininga": "CRAS - Piratininga",
    "CRAS - Munhoz": "CRAS - Munhoz Jr.",
    "CRAS - Munhoz Júnior": "CRAS - Munhoz Jr.",
    "CRAS - Jardim D'Abril": "CRAS - Jardim D'Abril",
    "CRAS 1º de Maio": "CRAS - 1º de Maio",
    "CRAS Bonança": "CRAS - Bonança",
    "CRAS Km 18": "CRAS - Km 18",
    "CRAS Munhoz Jr.": "CRAS - Munhoz Jr.",
    "CRAS Padroeira": "CRAS - Padroeira",
    "CRAS Piratininga": "CRAS - Piratininga",
    "CRAS Rochdale": "CRAS - Rochdale",
    "CRAS Santo Antonio": "CRAS - Santo Antonio",
    "CRAS Veloso": "CRAS - Veloso",
}

StatementMeta(, d2a22d00-e133-4105-a6c5-c1b29694f76b, 4, Finished, Available, Finished, False)

## Funções Utilitárias

Funções reutilizáveis para normalização de colunas e conversão segura de tipos.

In [3]:
def normalizar_colunas(df):
    """Converte nomes de colunas para snake_case sem acentos."""
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[:()+,]", "", regex=True)
        .str.replace(r"[\s./']", "_", regex=True)
    )
    df.columns = [unidecode(col) for col in df.columns]
    return df


def safe_to_int(series, fill_value=0):
    """Converte série para int, tratando nulos e strings inválidas."""
    return pd.to_numeric(series, errors="coerce").fillna(fill_value).astype(int)

StatementMeta(, d2a22d00-e133-4105-a6c5-c1b29694f76b, 5, Finished, Available, Finished, False)

## Leitura e Transformação

Leitura do CSV bruto, expansão de colunas JSON (blocos A–D) e padronização.

In [4]:
def ler_csv_rma(caminho="/lakehouse/default/Files/raw_sas_rma/bd_rma.csv"):
    """Lê o CSV bruto de RMA com tratamento de encoding e linhas inválidas."""
    return pd.read_csv(
        caminho,
        sep=";",
        encoding="utf-8-sig",
        engine="python",
        quotechar='"',
        doublequote=True,
        quoting=csv.QUOTE_MINIMAL,
        on_bad_lines="skip",
        dtype=str,
    )


def ajustar_colunas_json(rma_acto):
    """Expande colunas JSON (blocos A, B, C, D) em colunas individuais."""
    def clean_json_string(json_str):
        if pd.isna(json_str) or json_str is None:
            return {}
        cleaned = json_str.replace("_x000D_", "").replace("\n", " ").strip()
        try:
            return json.loads(cleaned)
        except (json.JSONDecodeError, TypeError):
            return {}

    json_columns = ["A", "B", "C", "D"]
    normalized_dfs = []

    for col in json_columns:
        cleaned_json = rma_acto[col].apply(clean_json_string)
        normalized_df = json_normalize(cleaned_json)
        normalized_df.columns = [f"{col}_{col_name}" for col_name in normalized_df.columns]
        normalized_dfs.append(normalized_df)

    df_final = pd.concat(
        [rma_acto.drop(columns=json_columns)] + normalized_dfs,
        axis=1,
    )
    return df_final


def tratar_rmas(rma_acto, map_sigla_unidade):
    """Normaliza colunas e padroniza nomes de CRAS."""
    rma_acto = normalizar_colunas(rma_acto)
    rma_acto["mes"] = rma_acto["mes"].astype(int)

    # .replace() mantém valores não mapeados inalterados (ao contrário de .map())
    rma_acto["sigla_da_unidade"] = rma_acto["sigla_da_unidade"].replace(map_sigla_unidade)

    return rma_acto

StatementMeta(, d2a22d00-e133-4105-a6c5-c1b29694f76b, 6, Finished, Available, Finished, False)

## Geração de Bases Analíticas

Funções que transformam a base tratada em tabelas longas (melt) para análise:
indicadores (seções A–D), raça, identidade de gênero e bairros.

In [5]:
RENAME_MAP_INDICADORES = {
    # ---------------------- SEÇÃO A ----------------------
    "a_1_total_de_familias_em_acompanhamento_pelo_paif": "A.1. Total de famílias em acompanhamento pelo PAIF",
    "a_2_novas_familias_inseridas_no_acompanhamento_do_paif_durante_o_mes_de_referencia": "A.2. Novas famílias inseridas no acompanhamento do PAIF durante o mês de referência",
    "a_4_total_de_familias_no_cras_cuja_situacao_de_trabalho_infantil_foi_identificada_no_mes_de_referencia": "A.4. Total de famílias no CRAS cuja situação de trabalho infantil foi identificada no mês de referência",
    "a_5_total_de_criancas_ou_adolescentes_em_situacao_de_trabalho_infantil_identificadas_no_mes_de_referencia": "A.5. Total de crianças ou adolescentes em situação de trabalho infantil identificadas no mês de referência",
    "a_3_total_de_familias_desligadas__soma_a_3_1__a_3_2__a_3_3__a_3_4": "A.3. Total de famílias desligadas (soma A.3.1, A.3.2, A.3.3, A.3.4)",
    "a_3_1_total_de__familias_desligadas_por_avaliacao_tecnica": "A.3.1. Total de famílias desligadas por avaliação técnica",
    "a_3_2_total_de__familias_desligadas_por_evasao_ou_recusa_da_familia": "A.3.2. Total de famílias desligadas por evasão ou recusa da família",
    "a_3_3_total_de_familias_desligadas_por_mudanca_de_regiao_ou_municipio": "A.3.3. Total de famílias desligadas por mudança de região ou município",
    "a_3_4_total_de_familia_desligadas_por_nao_localizacao": "A.3.4. Total de famílias desligadas por não localização",
    # ---------------------- SEÇÃO B ----------------------
    "b_1_familias_em_situacao_de_extrema_pobreza": "B.1. Famílias em situação de extrema pobreza",
    "b_10_familias_imigrantes": "B.10. Famílias imigrantes",
    "b_11_familias_quilombolas": "B.11. Famílias quilombolas",
    "b_12_familias_ciganas": "B.12. Famílias ciganas",
    "b_13__familias_indigenas": "B.13. Famílias indígenas",
    "b_14_familias_de_outros_povos_ou_comunidades_tradicionais": "B.14. Famílias de outros povos ou comunidades tradicionais",
    "b_2_familias_beneficiarias_do_programa_bolsa_familia_auxilio_brasil": "B.2. Famílias beneficiárias do Programa Bolsa Família/Auxílio Brasil",
    "b_3_familias_beneficiarias_do_programa_bolsa_familia_auxilio_brasil_em_descumprimento_de_condicionalidades": "B.3. Famílias beneficiárias do Programa Bolsa Família/Auxílio Brasil em descumprimento de condicionalidades",
    "b_5__familias_com_criancas_ou_adolescentes_em_situacao_de_trabalho_infantil": "B.5. Famílias com crianças ou adolescentes em situação de trabalho infantil",
    "b_6_familias_com_criancas_ou_adolescentes_em_servico_de_acolhimento": "B.6. Famílias com crianças ou adolescentes em serviço de acolhimento",
    "b_7__familias_beneficiarias_do_programa_nosso_futuro": "B.7. Famílias beneficiárias do Programa Nosso Futuro",
    "b_8_familias_com_pessoa_com_deficiencia": "B.8. Famílias com pessoa com deficiência",
    "b_9_familias_com_pessoa_que_fazem_uso_abusivo_de_alcool_e_ou_substancia_psicoativa": "B.9. Famílias com pessoa que faz uso abusivo de álcool e/ou substâncias psicoativas",
    "b_4_familias_com_membros_beneficiarios_do_bpc_idoso_e_pcd_soma_b4_1_b4_2": "B.4. Famílias com membros beneficiários do BPC (idoso e PCD) – soma B.4.1 e B.4.2",
    "b_4_1__familias_com_membros_beneficiarios_do_bpc_idoso": "B.4.1. Famílias com membros beneficiários do BPC Idoso",
    "b_4_2_familias_com_membros_beneficiarios_do_bpc_pcd": "B.4.2. Famílias com membros beneficiários do BPC PCD",
    # ---------------------- SEÇÃO C ----------------------
    "c_1_total_de_atendimentos_particularizados_realizados_no_mes_de_referencia": "C.1. Total de atendimentos particularizados realizados no mês de referência",
    "c_10_total_de_auxilio_vulnerabilidade_temporaria_concedidos_entregues": "C.10. Total de auxílios vulnerabilidade temporária concedidos/entregues",
    "c_11_total_de_beneficio_calamidade_publica_concedidos_entregues": "C.11. Total de benefícios de calamidade pública concedidos/entregues",
    "c_12_total_de_familias_beneficiarias_do_auxilio_aos_cidadaos_vitimas_de_desastres_naturais": "C.12. Total de famílias beneficiárias do auxílio a cidadãos vítimas de desastres naturais",
    "c_13_total_de_familias_beneficiarias_pelo_banco_de_alimentos_apos_encaminhamento": "C.13. Total de famílias beneficiárias pelo Banco de Alimentos após encaminhamento",
    "c_2_familias_encaminhadas_para_inclusao_no_cadastro_unico": "C.2. Famílias encaminhadas para inclusão no Cadastro Único",
    "c_3_familias_encaminhadas_para_atualizacao_cadastral_no_cadastro_unico": "C.3. Famílias encaminhadas para atualização cadastral no Cadastro Único",
    "c_5__familias_encaminhadas_para_o_creas": "C.5. Famílias encaminhadas para o CREAS",
    "c_6_visitas_domiciliares_realizadas": "C.6. Visitas domiciliares realizadas",
    "c_7_total_de_auxilios-natalidade_concedidos_entregues": "C.7. Total de auxílios-natalidade concedidos/entregues",
    "c_8_total_de_auxilios-funeral_concedidos_entregues": "C.8. Total de auxílios-funeral concedidos/entregues",
    "c_9_outros_beneficios_eventuais_concedidos_entregues": "C.9. Outros benefícios eventuais concedidos/entregues",
    "c_14_total_de_familias_beneficiarias_com_bpc_pcd_e_ou_idoso_apos_encaminhamento_soma_c_14_1__c_14_2": "C.14. Total de famílias beneficiárias com BPC (PCD e/ou Idoso) após encaminhamento (soma C.14.1 e C.14.2)",
    "c_14_1_total_de_familias_beneficiarias_com_bpc_idoso_apos_encaminhamento": "C.14.1. Total de famílias beneficiárias com BPC Idoso após encaminhamento",
    "c_14_2_total_de_familias_beneficiarias_com_bpc_pcd_apos_encaminhamento": "C.14.2. Total de famílias beneficiárias com BPC PCD após encaminhamento",
    "c_1_1_total_de_atendimento_particularizados_realizados_no_mes_de_referencia_para_familias_imigrantes_quilombolas_ciganas_indigenas_e_outros_povos_ou_comunidades_tradicionais": "C.1.1. Total de atendimentos particularizados para famílias imigrantes, quilombolas, ciganas, indígenas e outros povos/comunidades tradicionais",
    "c_1_10_total_de_familias_atendidas_-_indigenas": "C.1.10. Total de famílias atendidas – indígenas",
    "c_1_11_total_de_familias_atendidas_de_outros_povos_ou_comunidades_tradicionais": "C.1.11. Total de famílias atendidas – outros povos ou comunidades tradicionais",
    "c_1_2_total_de_familias_atendidas_no_paif_pela_equipe_tecnica_e_nao_inseridas_em_acompanhamento": "C.1.2. Total de famílias atendidas no PAIF pela equipe técnica e não inseridas em acompanhamento",
    "c_1_3_total_de_familias_atendidas_em_acompanhamento_no_paif_pela_equipe_tecnica": "C.1.3. Total de famílias atendidas em acompanhamento no PAIF pela equipe técnica",
    "c_1_4_total_de_familias_atendidas_do_programa_nosso_futuro": "C.1.4. Total de famílias atendidas do Programa Nosso Futuro",
    "c_1_5__total_de_familias_beneficiarias_com_bpc_pcd_e_ou_idoso_atendidas_no_mes_de_referencia_soma_c_1_5_1___c_1_5_2_": "C.1.5. Total de famílias beneficiárias com BPC (PCD e/ou Idoso) atendidas no mês de referência (soma C.1.5.1 e C.1.5.2)",
    "c_1_5_1_total_de_familias_beneficiarias_com_bpc_idoso_atendidas_no_mes_de_referencia": "C.1.5.1. Total de famílias beneficiárias com BPC Idoso atendidas no mês de referência",
    "c_1_5_2_total_de_familias_beneficiarias_com_bpc_pcd_atendidas_no_mes_de_referencia": "C.1.5.2. Total de famílias beneficiárias com BPC PCD atendidas no mês de referência",
    "c_1_6_total_de_familias_atendidas_do_auxilio_aos_cidadaos_vitimas_de_desastres_naturais": "C.1.6. Total de famílias atendidas do auxílio a cidadãos vítimas de desastres naturais",
    "c_1_7_total_de_familias_atendidas_-_imigrantes": "C.1.7. Total de famílias atendidas – imigrantes",
    "c_1_8_total_de_familias_atendidas_-_quilombolas": "C.1.8. Total de famílias atendidas – quilombolas",
    "c_1_9_total_de_familias_atendidas_-_ciganas": "C.1.9. Total de famílias atendidas – ciganas",
    "c_4_individuos_encaminhados_para_acesso_ao_bpc_idoso_e_pcd_soma_c_4_1___c_4_2_": "C.4. Indivíduos encaminhados para acesso ao BPC (Idoso e PCD) – soma C.4.1 e C.4.2",
    "c_4_1__individuos_encaminhados_para_o_acesso_ao_bpc_idoso": "C.4.1. Indivíduos encaminhados para acesso ao BPC Idoso",
    "c_4_2_individuos_encaminhados_para_o_acesso_ao_bpc_pcd": "C.4.2. Indivíduos encaminhados para acesso ao BPC PCD",
    "c_5_1_familias_encaminhadas_para_o_servico_de_convivencia_e_fortalecimento_de_vinculo": "C.5.1. Famílias encaminhadas para o Serviço de Convivência e Fortalecimento de Vínculos",
    "c_5_10_familias_encaminhadas_para_acesso_ao_auxilio_funeral": "C.5.10. Famílias encaminhadas para acesso ao auxílio funeral",
    "c_5_11_familias_encaminhadas_para_acesso_ao_auxilio_vulnerabilidade_temporaria": "C.5.11. Famílias encaminhadas para acesso ao auxílio vulnerabilidade temporária",
    "c_5_12_familias_encaminhadas_para_acesso_ao_beneficio_calamidade_publica": "C.5.12. Famílias encaminhadas para acesso ao benefício calamidade pública",
    "c_5_13_familias_encaminhas_para_acesso_a_outros_beneficios_eventuais": "C.5.13. Famílias encaminhadas para acesso a outros benefícios eventuais",
    "c_5_14_familias_encaminhadas_ao_fundo_social_de_solidariedade": "C.5.14. Famílias encaminhadas ao Fundo Social de Solidariedade",
    "c_5_2_familias_encaminhadas_para_o_acessuas_trabalho": "C.5.2. Famílias encaminhadas para o Acessuas Trabalho",
    "c_5_3_familias_encaminhadas_para_servicos_de_saude": "C.5.3. Famílias encaminhadas para serviços de saúde",
    "c_5_4_familias_encaminhadas_para_servicos_de_trabalho_e_renda": "C.5.4. Famílias encaminhadas para serviços de trabalho e renda",
    "c_5_5_familias_encaminhadas_para_servicos_da_educacao": "C.5.5. Famílias encaminhadas para serviços da educação",
    "c_5_6_familias_encaminhadas_para_acesso_a_transporte": "C.5.6. Famílias encaminhadas para acesso a transporte",
    "c_5_7_familias_encaminhadas_para_acesso_ao_banco_de_alimentos": "C.5.7. Famílias encaminhadas para acesso ao Banco de Alimentos",
    "c_5_8_familias_encaminhadas_para_acesso_ao_programa_nosso_futuro": "C.5.8. Famílias encaminhadas para acesso ao Programa Nosso Futuro",
    "c_5_9_familias_encaminhadas_para_acesso_ao_auxilio_natalidade": "C.5.9. Famílias encaminhadas para acesso ao auxílio natalidade",
    # ---------------------- SEÇÃO D ----------------------
    "d_1_familias_participando_regularmente_de_grupos_no_ambito_do_paif": "D.1. Famílias participando regularmente de grupos no âmbito do PAIF",
    "d_2_criancas_de_0_a_6_anos_em_servicos_de_convivencia_e_fortalecimento_de_vinculos": "D.2. Crianças de 0 a 6 anos em Serviços de Convivência e Fortalecimento de Vínculos",
    "d_3_criancas_adolescentes_de_7_a_14_anos_em_servicos_de_convivencia_e_fortalecimento_de_vinculos": "D.3. Crianças e adolescentes de 7 a 14 anos em Serviços de Convivência e Fortalecimento de Vínculos",
    "d_4_adolescentes_de_15_a_17_anos_em_servicos_de_convivencia_e_fortalecimento_de_vinculos": "D.4. Adolescentes de 15 a 17 anos em Serviços de Convivência e Fortalecimento de Vínculos",
    "d_5_idosos_em_servicos_de_convivencia_e_fortalecimento_de_vinculos_para_idosos": "D.5. Idosos em Serviços de Convivência e Fortalecimento de Vínculos para Idosos",
    "d_6_pessoas_que_participaram_de_palestras_oficinas_e_outras_atividades_coletivas_de_carater_nao_continuado": "D.6. Pessoas que participaram de palestras, oficinas e outras atividades coletivas de caráter não continuado",
    "d_7__pessoas_com_deficiencia_participando_dos_servicos_de_convivencia_ou_dos_grupos_do_paif": "D.7. Pessoas com deficiência participando dos Serviços de Convivência ou dos grupos do PAIF",
    "d_8__adultos_entre_18_e_59_anos_em_servicos_de_convivencia_e_fortalecimento_de_vinculos": "D.8. Adultos entre 18 e 59 anos em Serviços de Convivência e Fortalecimento de Vínculos",
}


def gerar_base_indicadores(rma_acto_tratado, cols_id, rename_map):
    """Gera base de indicadores (seções A-D) no formato longo."""
    cols_indicadores = rma_acto_tratado.filter(regex="^(a_|b_|c_|d_)").columns.tolist()

    indicadores = rma_acto_tratado[cols_id + cols_indicadores].melt(
        id_vars=cols_id,
        value_vars=cols_indicadores,
        var_name="indicador",
        value_name="valor",
    )
    indicadores["indicador"] = indicadores["indicador"].replace(rename_map)
    return indicadores

StatementMeta(, d2a22d00-e133-4105-a6c5-c1b29694f76b, 7, Finished, Available, Finished, False)

In [6]:
def _extrair_raca(nome_coluna):
    """Extrai a raça a partir do nome da coluna normalizada."""
    nome = nome_coluna.lower()
    if "parda" in nome:
        return "Parda"
    elif "preta" in nome:
        return "Preta"
    elif "amarela" in nome:
        return "Amarela"
    elif "branca" in nome:
        return "Branca"
    elif "indigena" in nome:
        return "Indígena"
    return None


def gerar_base_raca(rma_acto_tratado, cols_id, cols_raca):
    """Gera base de cor/raça do RF (bloco E.1) no formato longo."""
    raca = (
        rma_acto_tratado[cols_id + cols_raca]
        .copy()
        .melt(
            id_vars=cols_id,
            value_vars=cols_raca,
            var_name="raca",
            value_name="valor",
        )
    )
    raca["genero"] = np.where(
        raca["raca"].str.contains("fem"), "Feminino", "Masculino"
    )
    raca["raca"] = raca["raca"].apply(_extrair_raca)
    return raca


DICT_ID_GENERO = {
    "homem_cis": "Homem Cis",
    "homem_trans": "Homem Trans",
    "intersexual": "Intersexual",
    "mulher_cis": "Mulher Cis",
    "mulher_trans": "Mulher Trans",
    "nao_binario": "Não binário",
    "nao_declarado": "Não declarado",
    "travesti": "Travesti",
}


def gerar_base_identidade_genero(rma_acto_tratado, cols_id, cols_id_genero, dict_id_genero):
    """Gera base de identidade de gênero do RF (bloco E.2) no formato longo."""
    identidade_genero = (
        rma_acto_tratado[cols_id + cols_id_genero]
        .copy()
        .melt(
            id_vars=cols_id,
            value_vars=cols_id_genero,
            var_name="identidade_genero",
            value_name="valor",
        )
    )
    identidade_genero["identidade_genero"] = identidade_genero["identidade_genero"].replace(dict_id_genero)
    return identidade_genero


def gerar_base_bairros(rma_acto_tratado, cols_id, cols_bairros):
    """Gera base de bairros de moradia (bloco F) no formato longo."""
    bairros = (
        rma_acto_tratado[cols_id + cols_bairros]
        .melt(
            id_vars=cols_id,
            value_vars=cols_bairros,
            var_name="bairro",
            value_name="valor",
        )
    )
    bairros["bairro"] = bairros["bairro"].str.replace("_", " ").str.title()
    return bairros

StatementMeta(, d2a22d00-e133-4105-a6c5-c1b29694f76b, 8, Finished, Available, Finished, False)

## Execução do Pipeline Principal

Lê os dados, aplica transformações e gera as 4 bases analíticas.

In [7]:
def main():
    rma_acto_origem = ler_csv_rma()
    # rma_acto = ajustar_colunas_json(rma_acto_origem)
    rma_acto_tratado = tratar_rmas(rma_acto_origem, MAP_SIGLA_UNIDADE)

    # Validação: verificar se houve CRAS perdidos no mapeamento
    nulos_sigla = rma_acto_tratado["sigla_da_unidade"].isna().sum()
    if nulos_sigla > 0:
        print(f"AVISO: {nulos_sigla} registros com sigla_da_unidade nula após mapeamento.")
        valores_unicos = rma_acto_origem["sigla_da_unidade"].unique()
        mapeados = set(MAP_SIGLA_UNIDADE.keys())
        nao_mapeados = [v for v in valores_unicos if v not in mapeados and pd.notna(v)]
        if nao_mapeados:
            print(f"  Valores não mapeados: {nao_mapeados}")

    indicadores = gerar_base_indicadores(rma_acto_tratado, COLS_ID, RENAME_MAP_INDICADORES)
    raca = gerar_base_raca(rma_acto_tratado, COLS_ID, COLS_RACA)
    identidade_genero = gerar_base_identidade_genero(rma_acto_tratado, COLS_ID, COLS_ID_GENERO, DICT_ID_GENERO)
    bairros = gerar_base_bairros(rma_acto_tratado, COLS_ID, COLS_BAIRROS)

    for df in [indicadores, raca, identidade_genero, bairros]:
        df["valor"] = safe_to_int(df["valor"])

    print(f"Indicadores: {indicadores.shape}")
    print(f"Raça: {raca.shape}")
    print(f"Identidade de gênero: {identidade_genero.shape}")
    print(f"Bairros: {bairros.shape}")

    return rma_acto_origem, rma_acto_tratado, indicadores, raca, identidade_genero, bairros


rma_acto_origem, rma_acto_tratado, indicadores, raca, identidade_genero, bairros = main()

StatementMeta(, d2a22d00-e133-4105-a6c5-c1b29694f76b, 9, Finished, Available, Finished, False)

Indicadores: (11154, 5)
Raça: (1430, 6)
Identidade de gênero: (1144, 5)
Bairros: (8580, 5)


## Inclusão de Dados Históricos

Incorpora dados de anos anteriores (2016–2024) a partir de arquivos Parquet e Excel do lakehouse.

In [8]:
def incluir_historico_raca(raca, cols_id, cols_raca, caminho_excel):
    """Inclui dados históricos de raça a partir de planilha Excel."""
    e2024 = pd.read_excel(caminho_excel)
    e2024 = normalizar_colunas(e2024)
    e2024 = gerar_base_raca(e2024, cols_id, cols_raca)

    raca = pd.concat([raca, e2024], axis=0, ignore_index=True)

    raca = raca.astype({
        "ano": int,
        "mes": int,
        "sigla_da_unidade": str,
        "raca": str,
        "valor": float,
        "genero": str,
    })
    return raca


raca = incluir_historico_raca(
    raca, COLS_ID, COLS_RACA,
    "/lakehouse/default/Files/raw_sas_rma/estrutura_bloco_e.xlsx",
)

StatementMeta(, d2a22d00-e133-4105-a6c5-c1b29694f76b, 10, Finished, Available, Finished, False)

In [9]:
def incluir_historico_indicadores(indicadores, diretorio_parquets):
    """Inclui histórico de indicadores lendo todos os parquets do diretório via glob."""
    padrao = f"{diretorio_parquets}/rma_cras_*_carga_indicadores.parquet"
    arquivos = sorted(glob_module.glob(padrao))

    if not arquivos:
        print(f"AVISO: Nenhum arquivo encontrado em '{padrao}'")
        return indicadores

    print(f"Carregando {len(arquivos)} arquivo(s) de histórico:")
    dfs = []
    for arquivo in arquivos:
        df = pd.read_parquet(arquivo)
        print(f"  {os.path.basename(arquivo)}: {len(df)} registros")
        dfs.append(df)

    indicadores_total = pd.concat(
        [indicadores] + dfs,
        axis=0,
        ignore_index=True,
    )
    indicadores_total = indicadores_total.astype({
        "ano": int,
        "mes": int,
        "sigla_da_unidade": str,
        "indicador": str,
        "valor": float,
    })
    indicadores_total['sigla_da_unidade'] = indicadores_total['sigla_da_unidade'].replace(MAP_SIGLA_UNIDADE)
    return indicadores_total


indicadores_total = incluir_historico_indicadores(
    indicadores,
    "/lakehouse/default/Files/raw_sas_rma",
)

StatementMeta(, d2a22d00-e133-4105-a6c5-c1b29694f76b, 11, Finished, Available, Finished, False)

Carregando 9 arquivo(s) de histórico:
  rma_cras_2016_carga_indicadores.parquet: 2592 registros
  rma_cras_2017_carga_indicadores.parquet: 3024 registros
  rma_cras_2018_carga_indicadores.parquet: 3024 registros
  rma_cras_2019_carga_indicadores.parquet: 3024 registros
  rma_cras_2020_carga_indicadores.parquet: 3024 registros
  rma_cras_2021_carga_indicadores.parquet: 3024 registros
  rma_cras_2022_carga_indicadores.parquet: 8772 registros
  rma_cras_2023_carga_indicadores.parquet: 8748 registros
  rma_cras_2024_carga_indicadores.parquet: 8832 registros


In [10]:
CONSOLIDACAO_INDICADORES = {
    # A.2
    "A.2. Novas famílias inseridas no acompanhamento": "A.2. Novas famílias inseridas no acompanhamento",
    "A.2. Novas famílias inseridas no acompanhamento do PAIF durante o mês": "A.2. Novas famílias inseridas no acompanhamento",
    "A.2. Novas famílias inseridas no acompanhamento do PAIF durante o mês de referência": "A.2. Novas famílias inseridas no acompanhamento",
    # A.3
    "A.3. Total de famílias cujo acompanhamento foi encerrado": "A.3. Total de famílias cujo acompanhamento foi encerrado",
    "A.3. Total de famílias desligadas (soma A.3.1, A.3.2, A.3.3, A.3.4)": "A.3. Total de famílias cujo acompanhamento foi encerrado",
    # A.4
    "A.4. Total de famílias cuja situação de trabalho infantil foi identificada no mês de referência": "A.4. Total de famílias cuja situação de trabalho infantil foi identificada no mês de referência",
    "A.4. Total de famílias no CRAS cuja situação de trabalho infantil foi identificada no mês de referência": "A.4. Total de famílias cuja situação de trabalho infantil foi identificada no mês de referência",
    # B.2
    "B.2. Famílias beneficiárias do Programa Bolsa Família": "B.2. Famílias beneficiárias do Programa Bolsa Família",
    "B.2. Famílias beneficiárias do Programa Bolsa Família/Auxílio Brasil": "B.2. Famílias beneficiárias do Programa Bolsa Família",
    # B.3
    "B.3. Famílias beneficiárias do Programa Bolsa Família, em descumprimento de condicionalidades": "B.3. Famílias beneficiárias do Programa Bolsa Família, em descumprimento de condicionalidades",
    "B.3. Famílias beneficiárias do Programa Bolsa Família/Auxílio Brasil em descumprimento de condicionalidades": "B.3. Famílias beneficiárias do Programa Bolsa Família, em descumprimento de condicionalidades",
    # B.4
    "B.4. Famílias com membros beneficiários do BPC": "B.4. Famílias com membros beneficiários do BPC",
    "B.4. Famílias com membros beneficiários do BPC (Idoso e PcD)": "B.4. Famílias com membros beneficiários do BPC",
    "B.4. Famílias com membros beneficiários do BPC (idoso e PCD) – soma B.4.1 e B.4.2": "B.4. Famílias com membros beneficiários do BPC",
    # B.4.2
    "B.4.2. Famílias com membros beneficiários do BPC PCD": "B.4.2. Famílias com membros beneficiários do BPC PCD",
    "B.4.2. Famílias com membros beneficiários do BPC PcD": "B.4.2. Famílias com membros beneficiários do BPC PCD",
    # B.6
    "B.6. Famílias com crianças ou adolescentes em Serviço de Acolhimento": "B.6. Famílias com crianças ou adolescentes em Serviço de Acolhimento",
    "B.6. Famílias com crianças ou adolescentes em serviço de acolhimento": "B.6. Famílias com crianças ou adolescentes em Serviço de Acolhimento",
    # C.1
    "C.1. Total de atendimentos individualizados realizados, por mês": "C.1. Total de atendimentos individualizados realizados, por mês",
    "C.1. Total de atendimentos particularizados realizados no mês de referência": "C.1. Total de atendimentos individualizados realizados, por mês",
    "C.1. Total de atendimentos particularizados realizados, por mês": "C.1. Total de atendimentos individualizados realizados, por mês",
    # C.1.10
    "C.1.10. Total de famílias atendidas - indígenas": "C.1.10. Total de famílias atendidas - indígenas",
    "C.1.10. Total de famílias atendidas – indígenas": "C.1.10. Total de famílias atendidas - indígenas",
    # C.1.5
    "C.1.5. Total de famílias beneficiárias com BPC (PCD e/ou Idoso) atendidas no mês de referência (soma C.1.5.1 e C.1.5.2)": "C.1.5. Total de famílias beneficiárias com BPC (PCD e/ou Idoso) atendidas no mês de referência (soma C.1.5.1 e C.1.5.2)",
    "C.1.5. Total de famílias beneficiárias de BPC (Idoso e/ou PcD) atendidas no mês de referência": "C.1.5. Total de famílias beneficiárias com BPC (PCD e/ou Idoso) atendidas no mês de referência (soma C.1.5.1 e C.1.5.2)",
    # C.1.5.1
    "C.1.5.1. Total de famílias beneficiárias com BPC Idoso atendidas no mês de referência": "C.1.5.1. Total de famílias beneficiárias com BPC Idoso atendidas no mês de referência",
    "C.1.5.1. Total de famílias beneficiárias do BPC Idoso atendidas no mês de referência": "C.1.5.1. Total de famílias beneficiárias com BPC Idoso atendidas no mês de referência",
    # C.1.5.2
    "C.1.5.2. Total de famílias beneficiárias com BPC PCD atendidas no mês de referência": "C.1.5.2. Total de famílias beneficiárias com BPC PCD atendidas no mês de referência",
    "C.1.5.2. Total de famílias beneficiárias do BPC PcD atendidas no mês de referência": "C.1.5.2. Total de famílias beneficiárias com BPC PCD atendidas no mês de referência",
    # C.1.6
    "C.1.6. Total de famílias atendidas do auxílio a cidadãos vítimas de desastres naturais": "C.1.6. Total de famílias atendidas do auxílio a cidadãos vítimas de desastres naturais",
    "C.1.6. Total de famílias atendidas pelo auxílio aos cidadãos vítimas de desastres naturais": "C.1.6. Total de famílias atendidas do auxílio a cidadãos vítimas de desastres naturais",
    # C.1.7
    "C.1.7. Total de famílias atendidas - imigrantes": "C.1.7. Total de famílias atendidas - imigrantes",
    "C.1.7. Total de famílias atendidas – imigrantes": "C.1.7. Total de famílias atendidas - imigrantes",
    # C.1.8
    "C.1.8. Total de famílias atendidas - quilombolas": "C.1.8. Total de famílias atendidas - quilombolas",
    "C.1.8. Total de famílias atendidas – quilombolas": "C.1.8. Total de famílias atendidas - quilombolas",
    # C.1.9
    "C.1.9. Total de famílias atendidas - ciganas": "C.1.9. Total de famílias atendidas - ciganas",
    "C.1.9. Total de famílias atendidas – ciganas": "C.1.9. Total de famílias atendidas - ciganas",
    # C.10
    "C.10. Total de auxílio vulnerabilidade temporária concedidos/entregues": "C.10. Total de auxílio vulnerabilidade temporária concedidos/entregues",
    "C.10. Total de auxílios vulnerabilidade temporária concedidos/entregues": "C.10. Total de auxílio vulnerabilidade temporária concedidos/entregues",
    # C.11
    "C.11. Total de benefícios calamidade pública concedidos/entregues": "C.11. Total de benefícios calamidade pública concedidos/entregues",
    "C.11. Total de benefícios de calamidade pública concedidos/entregues": "C.11. Total de benefícios calamidade pública concedidos/entregues",
    # C.12
    "C.12. Total de famílias beneficiárias do auxílio a cidadãos vítimas de desastres naturais": "C.12. Total de famílias beneficiárias do auxílio a cidadãos vítimas de desastres naturais",
    "C.12. Total de famílias beneficiárias do auxílio aos cidadãos vítimas de desastres naturais": "C.12. Total de famílias beneficiárias do auxílio a cidadãos vítimas de desastres naturais",
    # C.14
    "C.14. Total de famílias beneficiárias com BPC (PCD e/ou Idoso) após encaminhamento (soma C.14.1 e C.14.2)": "C.14. Total de famílias beneficiárias com BPC (PCD e/ou Idoso) após encaminhamento (soma C.14.1 e C.14.2)",
    "C.14. Total de famílias beneficiárias com BPC Idoso após encaminhamento": "C.14. Total de famílias beneficiárias com BPC (PCD e/ou Idoso) após encaminhamento (soma C.14.1 e C.14.2)",
    "C.14. Total de famílias beneficiárias com BPC PcD e/ou Idoso após encaminhamento": "C.14. Total de famílias beneficiárias com BPC (PCD e/ou Idoso) após encaminhamento (soma C.14.1 e C.14.2)",
    # C.14.2
    "C.14.2. Total de famílias beneficiárias com BPC PCD após encaminhamento": "C.14.2. Total de famílias beneficiárias com BPC PCD após encaminhamento",
    "C.14.2. Total de famílias beneficiárias com BPC PcD após encaminhamento": "C.14.2. Total de famílias beneficiárias com BPC PCD após encaminhamento",
    # C.2
    "C.2. Famílias encaminhadas para inclusão no Cadastro Único": "C.2. Famílias encaminhadas para inclusão no Cadastro Único",
    "C.2. Famílias encaminhadas para inclusão no Cadastro Único, por mês": "C.2. Famílias encaminhadas para inclusão no Cadastro Único",
    # C.4
    "C.4. Indivíduos encaminhados para acesso ao BPC": "C.4. Indivíduos encaminhados para acesso ao BPC",
    "C.4. Indivíduos encaminhados para acesso ao BPC (Idoso e PCD) – soma C.4.1 e C.4.2": "C.4. Indivíduos encaminhados para acesso ao BPC",
    "C.4. Indivíduos encaminhados para acesso ao BPC (Idoso e/ou PcD)": "C.4. Indivíduos encaminhados para acesso ao BPC",
    # C.4.1
    "C.4.1. Indivíduos encaminhados para acesso ao BPC Idoso": "C.4.1. Indivíduos encaminhados para acesso ao BPC Idoso",
    "C.4.1. Indivíduos encaminhados para o acesso ao BPC Idoso": "C.4.1. Indivíduos encaminhados para acesso ao BPC Idoso",
    # C.4.2
    "C.4.2. Indivíduos encaminhados para acesso ao BPC PCD": "C.4.2. Indivíduos encaminhados para acesso ao BPC PCD",
    "C.4.2. Indivíduos encaminhados para o acesso ao BPC PcD": "C.4.2. Indivíduos encaminhados para acesso ao BPC PCD",
    # C.5.2
    "C.5.2. Famílias encaminhadas para o ACESSUAS Trabalho": "C.5.2. Famílias encaminhadas para o ACESSUAS Trabalho",
    "C.5.2. Famílias encaminhadas para o Acessuas Trabalho": "C.5.2. Famílias encaminhadas para o ACESSUAS Trabalho",
    # C.5.3
    "C.5.3. Famílias encaminhadas para serviços de Saúde": "C.5.3. Famílias encaminhadas para serviços de Saúde",
    "C.5.3. Famílias encaminhadas para serviços de saúde": "C.5.3. Famílias encaminhadas para serviços de Saúde",
    # C.5.4
    "C.5.4. Famílias encaminhadas para serviços de Trabalho e Renda": "C.5.4. Famílias encaminhadas para serviços de Trabalho e Renda",
    "C.5.4. Famílias encaminhadas para serviços de trabalho e renda": "C.5.4. Famílias encaminhadas para serviços de Trabalho e Renda",
    # C.5.5
    "C.5.5. Famílias encaminhadas para serviços da educação": "C.5.5. Famílias encaminhadas para serviços da educação",
    "C.5.5. Famílias encaminhadas para serviços de Educação": "C.5.5. Famílias encaminhadas para serviços da educação",
    # C.5.6
    "C.5.6. Famílias encaminhadas para acesso a Transporte": "C.5.6. Famílias encaminhadas para acesso a Transporte",
    "C.5.6. Famílias encaminhadas para acesso a transporte": "C.5.6. Famílias encaminhadas para acesso a Transporte",
    # D.1
    "D.1. Famílias participando regularmente de grupos no âmbito do PAIF": "D.1. Famílias participando regularmente de grupos no âmbito do PAIF",
    "D.1. Famílias participando regularmente do grupo PAIF": "D.1. Famílias participando regularmente de grupos no âmbito do PAIF",
    # D.3
    "D.3. Crianças e adolescentes de 7 a 14 anos em Serviços de Convivência e Fortalecimento de Vínculos": "D.3. Crianças e adolescentes de 7 a 14 anos em Serviços de Convivência e Fortalecimento de Vínculos",
    "D.3. Crianças/adolescentes de 7 a 14 anos em Serviços de Convivência e Fortalecimento de Vínculos": "D.3. Crianças e adolescentes de 7 a 14 anos em Serviços de Convivência e Fortalecimento de Vínculos",
    # D.5
    "D.5. Idosos em Serviços de Convivência e Fortalecimento de Vínculos para Idosos": "D.5. Idosos em Serviços de Convivência e Fortalecimento de Vínculos para Idosos",
    "D.5. Idosos em Serviços de Convivência e Fortalecimentos de Vínculos": "D.5. Idosos em Serviços de Convivência e Fortalecimento de Vínculos para Idosos",
    # D.7
    "D.7. Pessoas com deficiência participando dos Serviços de Convivência ou dos grupos do PAIF": "D.7. Pessoas com deficiência participando dos Serviços de Convivência ou dos grupos do PAIF",
    "D.7. Pessoas com deficiência, participando dos Serviços de Convivência ou dos grupos do PAIF": "D.7. Pessoas com deficiência participando dos Serviços de Convivência ou dos grupos do PAIF",
}

indicadores_total["indicador"] = indicadores_total["indicador"].replace(CONSOLIDACAO_INDICADORES)
print(f"Indicadores com histórico: {indicadores_total.shape}")

indicadores_total["indicador_media"] = indicadores_total["indicador"].replace(
    "A.1. Total de famílias em acompanhamento pelo PAIF",
    "A.1. Média de famílias em acompanhamento pelo PAIF",
)
indicadores_total = indicadores_total.loc[~indicadores_total['indicador'].str.contains("Fonte: ")].copy()

StatementMeta(, d2a22d00-e133-4105-a6c5-c1b29694f76b, 12, Finished, Available, Finished, False)

Indicadores com histórico: (55218, 5)


## Tipagem Final e Validações

Garante tipos corretos nas bases históricas e executa verificações de integridade.

In [11]:
identidade_genero = identidade_genero.astype({
    "ano": int,
    "mes": int,
    "sigla_da_unidade": str,
    "identidade_genero": str,
})

bairros = bairros.astype({
    "ano": int,
    "mes": int,
    "sigla_da_unidade": str,
    "bairro": str,
    "valor": int,
})

StatementMeta(, d2a22d00-e133-4105-a6c5-c1b29694f76b, 13, Finished, Available, Finished, False)

In [12]:
def validar_dados(rma_acto_tratado, indicadores, raca, identidade_genero, bairros):
    """Executa verificações de integridade nos dados processados."""
    erros = []

    # 1. Verificar NaN em sigla_da_unidade
    for nome, df in [("indicadores", indicadores), ("raca", raca),
                     ("identidade_genero", identidade_genero), ("bairros", bairros)]:
        n_nulos = df["sigla_da_unidade"].isna().sum()
        if n_nulos > 0:
            erros.append(f"  [{nome}] {n_nulos} registros com sigla_da_unidade nula")

    # 2. Verificar se raça tem valores None (indica bug na extração)
    if raca["raca"].isna().any():
        n_nulos_raca = raca["raca"].isna().sum()
        erros.append(f"  [raca] {n_nulos_raca} registros com raça não identificada (None)")

    # 3. Verificar subtotais A.3 = A.3.1 + A.3.2 + A.3.3 + A.3.4
    cols_a3 = [c for c in rma_acto_tratado.columns if c.startswith("a_3")]
    if len(cols_a3) >= 5:
        col_total = [c for c in cols_a3 if "soma" in c or (c.count("_") <= 6 and "total" in c)]
        col_parciais = [c for c in cols_a3 if c.startswith("a_3_1") or c.startswith("a_3_2")
                        or c.startswith("a_3_3") or c.startswith("a_3_4")]
        if col_total and len(col_parciais) == 4:
            total_col = col_total[0]
            soma_parciais = sum(
                pd.to_numeric(rma_acto_tratado[c], errors="coerce").fillna(0)
                for c in col_parciais
            )
            total_vals = pd.to_numeric(rma_acto_tratado[total_col], errors="coerce").fillna(0)
            divergencias = (total_vals != soma_parciais).sum()
            if divergencias > 0:
                erros.append(f"  [A.3] {divergencias} registros onde A.3 != A.3.1 + A.3.2 + A.3.3 + A.3.4")

    # 4. Verificar valores negativos
    for nome, df in [("indicadores", indicadores), ("raca", raca), ("bairros", bairros)]:
        n_negativos = (df["valor"] < 0).sum()
        if n_negativos > 0:
            erros.append(f"  [{nome}] {n_negativos} registros com valores negativos")

    if erros:
        print("VALIDAÇÃO - Problemas encontrados:")
        for e in erros:
            print(e)
    else:
        print("VALIDAÇÃO - Todos os checks passaram com sucesso.")


validar_dados(rma_acto_tratado, indicadores, raca, identidade_genero, bairros)

StatementMeta(, d2a22d00-e133-4105-a6c5-c1b29694f76b, 14, Finished, Available, Finished, False)

VALIDAÇÃO - Todos os checks passaram com sucesso.


## Escrita no lakehouse

In [13]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
)

schema_indicadores = StructType(
    [
        StructField("ano", IntegerType(), True),
        StructField("mes", IntegerType(), True),
        StructField("sigla_da_unidade", StringType(), True),
        StructField("indicador", StringType(), True),
        StructField("valor", DoubleType(), True),
        StructField("indicador_media", StringType(), True),
    ]
)
indicadores_spark = spark.createDataFrame(indicadores_total, schema=schema_indicadores)
indicadores_spark.write.mode("overwrite").format("delta").option(
    "overwriteSchema", "true"
).saveAsTable("gold_rma_cras_indicadores")


schema_raca = StructType(
    [
        StructField("ano", IntegerType(), True),
        StructField("mes", IntegerType(), True),
        StructField("sigla_da_unidade", StringType(), True),
        StructField("raca", StringType(), True),
        StructField("valor", IntegerType(), True),
        StructField("genero", StringType(), True),
    ]
)
raca_spark = spark.createDataFrame(raca, schema=schema_raca)
raca_spark.write.mode("overwrite").format("delta").option(
    "overwriteSchema", "true"
).saveAsTable("gold_rma_cras_raca_cor")


schema_id_genero = StructType(
    [
        StructField("ano", IntegerType(), True),
        StructField("mes", IntegerType(), True),
        StructField("sigla_da_unidade", StringType(), True),
        StructField("identidade_genero", StringType(), True),
        StructField("valor", IntegerType(), True),
    ]
)
identidade_genero_spark = spark.createDataFrame(
    identidade_genero, schema=schema_id_genero
)
identidade_genero_spark.write.mode("overwrite").format("delta").option(
    "overwriteSchema", "true"
).saveAsTable("gold_rma_cras_id_genero")


schema_bairros = StructType(
    [
        StructField("ano", IntegerType(), True),
        StructField("mes", IntegerType(), True),
        StructField("sigla_da_unidade", StringType(), True),
        StructField("bairro", StringType(), True),
        StructField("valor", IntegerType(), True),
    ]
)
bairros_spark = spark.createDataFrame(bairros, schema=schema_bairros)
bairros_spark.write.mode("overwrite").format("delta").option(
    "overwriteSchema", "true"
).saveAsTable("gold_rma_cras_bairros")

StatementMeta(, d2a22d00-e133-4105-a6c5-c1b29694f76b, 15, Finished, Available, Finished, False)